# Introduction aux LLMs — GPT-2 Text Generation
**Notebook complet — Exercices 1 à 5**

Modèle utilisé : `gpt2` (HuggingFace Transformers)

## Setup — Installation des librairies

In [ ]:
# Install necessary libraries
!pip install transformers matplotlib --quiet

# Import required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Librairies importees avec succes.')

## Exercice 1 : What are Large Language Models (LLMs) ?

### 1. Definition

Un **Large Language Model (LLM)** est un modele de deep learning entraine sur d'enormes quantites de texte (livres, sites web, code, articles...) pour apprendre les structures statistiques du langage naturel.

Concretement, un LLM apprend a **predire le mot suivant** dans une sequence, ce qui lui permet ensuite de generer du texte coherent, de repondre a des questions, de resumer, de traduire, ou d'ecrire du code.

Exemples connus : GPT-4 (OpenAI), Gemini (Google), Claude (Anthropic), LLaMA (Meta).

### 2. Chargement du modele GPT-2

In [ ]:
# 2. Loading a pretrained model and tokenizer
model_name = 'gpt2'  # GPT-2 utilise ici pour demonstration

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence.
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")

## Exercice 2 : Transformer Architecture and Tokenization

### 1. La tokenisation — Explication

La **tokenisation** est l'etape qui transforme un texte brut en une sequence de **tokens**, c'est-a-dire les unites elementaires que le modele peut traiter.

Un token n'est pas toujours un mot entier : il peut etre un mot complet, un sous-mot, un signe de ponctuation, ou meme un espace. Par exemple, le mot `playing` peut etre decoupage en `play` + `ing`.

Chaque token est ensuite converti en un **identifiant numerique unique** (token ID) via le vocabulaire du modele. C'est sur ces IDs que le modele travaille, pas sur le texte brut.

### 2 & 3. Tokenisation et visualisation

In [ ]:
# 1. Phrase de test
text = 'Artificial intelligence is transforming the world rapidly.'

# 2. Tokenize input text
tokens    = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f'Original Text : {text}')
print(f'Tokens        : {tokens}')
print(f'Token IDs     : {token_ids}')

# 3. Visualizing the tokenization process
plt.figure(figsize=(12, 5))
bars = plt.bar(tokens, token_ids, color='skyblue', edgecolor='white', linewidth=0.8)
for bar, tid in zip(bars, token_ids):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             str(tid), ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.xlabel('Tokens')
plt.ylabel('Token IDs')
plt.title('Tokenisation de la phrase — Token vs Token ID')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Exercice 3 : Token IDs and Special Prefixes

### 1. Affichage de l'ID de chaque token

In [ ]:
# 1. Affichage token par token
print(f"{'Token':<20} {'ID'}")
print('-' * 30)
for token, tid in zip(tokens, token_ids):
    print(f"{token:<20} {tid}")

### 2. Le prefixe special `Ġ`

Le caractere `Ġ` (G avec point en dessous) est utilise par le tokenizer **BPE (Byte-Pair Encoding)** de GPT-2 pour indiquer qu'un token est **precede d'un espace** dans le texte original.

Autrement dit :
- `intelligence` sans `Ġ` = debut de texte ou collé au token precedent
- `Ġintelligence` avec `Ġ` = le mot est precede d'un espace (debut d'un nouveau mot)

Cela permet au modele de distinguer `'play'` dans `'gameplay'` (pas d'espace) de `'play'` dans `'let's play'` (avec espace avant). C'est essentiel pour que le modele comprenne la structure du texte au niveau des mots.

## Exercice 4 : Pretraining vs Fine-Tuning

### Pretraining

Le **pretraining** est la premiere phase d'entrainement, realisee sur d'enormes corpus de texte (des centaines de milliards de mots). Le modele apprend de maniere **non supervisee** a predire le token suivant dans une sequence. Il n'y a pas de tache specifique : le modele construit une representation generale du langage — grammaire, faits, raisonnement, style.

C'est une etape couteuse en temps et en ressources (semaines sur des milliers de GPUs), mais elle est faite une seule fois. GPT-2 a ete pre-entraine ainsi par OpenAI.

### Fine-Tuning

Le **fine-tuning** est la deuxieme phase. A partir du modele pre-entraine, on continue l'entrainement sur un **dataset specifique et plus petit**, oriente vers une tache precise : classification de sentiment, reponse a des questions, generation de code, suivi d'instructions, etc.

Le modele ajuste ses poids pour specialiser ses capacites generales sur la nouvelle tache, sans repartir de zero. C'est beaucoup moins couteux que le pretraining.

**Analogie** : le pretraining, c'est apprendre a lire et ecrire pendant 20 ans. Le fine-tuning, c'est suivre un stage de 3 mois pour devenir expert comptable.

## Exercice 5 : Generate Simple Text

### 1 & 2. Generation de texte avec GPT-2

In [ ]:
# 1. Phrase d'entree
input_text = 'The future of machine learning is'

# Tokenisation de l'entree
input_ids = tokenizer.encode(input_text, return_tensors='pt')

# 2. Generation : le modele predit les tokens suivants de maniere sequentielle
# max_new_tokens = nombre de tokens a generer apres le prompt
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=True,       # echantillonnage stochastique (plus creatif que greedy)
        temperature=0.8,      # controle la creativite (< 1 = plus previsible, > 1 = plus aleatoire)
        top_k=50,             # garde les 50 tokens les plus probables a chaque etape
        pad_token_id=tokenizer.eos_token_id
    )

# Decodage des IDs en texte lisible
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f'Input          : {input_text}')
print(f'Generated Output: {output_text}')

### Explication du processus de generation

Le modele genere le texte **token par token** :
1. Il recoit le prompt encode en token IDs
2. Il predit une **distribution de probabilite** sur tout le vocabulaire (~50 000 tokens)
3. Il echantillonne un token selon cette distribution
4. Il ajoute ce token au contexte et recommence

Les parametres `temperature`, `top_k` et `do_sample` controlent le compromis entre coherence et creativite de la generation.

## Bonus — Pipeline simplifie HuggingFace

In [ ]:
# HuggingFace fournit un pipeline de haut niveau qui simplifie tout
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

result = generator(
    'Deep learning has revolutionized',
    max_new_tokens=40,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    truncation=True
)
print(result[0]['generated_text'])